# Routiq AI Coach — interactive foundation

This follows the six-cell layout of your `chat_llm.ipynb` sample: environment → packages → configuration/secrets → client → interactive loop.

**Current mode: OFFLINE_FOUNDATION.** This notebook runs the existing FastAPI scaffold in memory. It does not call OpenAI, execute CrewAI agents or generate habit advice. Chat requests correctly return `FOUNDATION_ONLY` until later approved milestones. The synthetic identity below is a notebook test fixture, not Routiq authentication.

1. Select this project's **Python 3.12 `.venv`** in your editor's kernel picker. Do not use the sample's Python 3.14 kernel; CrewAI 1.15.27 requires Python >=3.10,<3.14.
2. Run the five code cells from top to bottom. Packages are checked, not automatically downloaded during notebook execution.
3. No API key is needed. The notebook never loads or prints `.env` values.
4. Type at `You:`. `/samples`, `/status`, `/help` and `/reset` inspect the foundation. `exit` or `quit` stops the loop. Rerunning the final cell starts fresh local history.
5. Outputs may contain your typed messages. Clear outputs before sharing.

Live responses will use the approved CrewAI Flow → authorized application services architecture when those milestones are implemented; this notebook does not bypass that design.

In [5]:
import sys
from importlib.metadata import PackageNotFoundError, version

print("Python path:", sys.executable)

for package in ("fastapi", "httpx", "pydantic", "crewai", "openai", "ipykernel"):
    try:
        print(package, version(package))
    except PackageNotFoundError:
        print(package, "MISSING")

Python path: C:\Users\ADMIN\AppData\Local\Python\pythoncore-3.14-64\python.exe
fastapi MISSING
httpx MISSING
pydantic MISSING
crewai MISSING
openai MISSING
ipykernel 7.4.0


In [6]:
import sys
print(sys.executable)

C:\Users\ADMIN\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [7]:
# 3. Load the preserved project configuration. Never load or display secret values.
from app.config.loader import load_config

coach_config = load_config(project_root / "app" / "config" / "routiq_chatbot.json")
env_path = project_root / ".env"
print("Configuration:", coach_config.document.version, coach_config.document.status)
print("Proposed questions:", len(coach_config.sample_questions))
print("Module .env:", "present but not loaded" if env_path.is_file() else "not required in offline mode")
print("API key access: disabled. Product decisions and thresholds remain unresolved.")

ModuleNotFoundError: No module named 'pydantic'

In [8]:
import sys
print(sys.executable)

C:\Users\ADMIN\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [ ]:
# 4. Create an in-memory FastAPI test client, not a direct OpenAI client.
from fastapi.testclient import TestClient
from app.main import create_app
from app.api.chat import Principal, authenticated_principal
from app.schemas.chat import ChatRequest

notebook_app = create_app(project_root / "app" / "config" / "routiq_chatbot.json")
# Explicit test-only identity: affects this in-memory app instance only.
notebook_app.dependency_overrides[authenticated_principal] = lambda: Principal("NOTEBOOK_SYNTHETIC_USER")
client = TestClient(notebook_app)
context = {"screen": "PROGRESS", "habit_id": None}
with client:
    assert client.get("/health").json()["live_ai_enabled"] is False
    print("Readiness:", client.get("/ready").json())
print("Client ready for offline boundary testing; authentication is synthetic.")

In [ ]:
# 5. Interactive loop. Local history is a notebook aid, not authoritative habit data.
from pydantic import ValidationError

local_history = []
print("Routiq foundation ready. Type /help for commands; exit or quit to stop.")
print("Live AI replies are unavailable at Milestone 1.")
with client:
    while True:
        try:
            message = input("You: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("Chat stopped.")
            break
        if message.lower() in {"exit", "quit", "/exit"}:
            print("Chat stopped.")
            break
        if not message:
            print("Enter a message, or exit to stop.")
            continue
        if message == "/help":
            print("/samples: proposed question examples; /status: API readiness; "
                  "/reset: clear local history; exit: stop.")
            continue
        if message == "/samples":
            for question in coach_config.sample_questions[:5]:
                print(f"{question.id}: {question.text} (proposed: {question.intent_id})")
            continue
        if message == "/status":
            print(client.get("/ready").json())
            continue
        if message == "/reset":
            local_history.clear()
            print("Local history cleared. No stored conversation exists.")
            continue
        try:
            request = ChatRequest(message=message, conversation_id=None, context=context)
        except ValidationError:
            print("Message or screen context is invalid. Use 1–4000 characters and a supported screen.")
            continue
        response = client.post("/api/v1/ai/chat", json=request.model_dump())
        if response.status_code == 503 and response.json().get("code") == "FOUNDATION_ONLY":
            reply = response.json()["message"]
        elif response.status_code == 401:
            reply = "Verified authentication is not integrated."
        else:
            reply = "Unexpected foundation response. Review the local implementation before continuing."
        local_history.append({"user": message, "foundation_response": reply})
        print("Routiq foundation:", reply)
print("Session ended; rerun this cell for fresh local history.")